# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [10]:
# Write your code below.
%load_ext dotenv
%dotenv 
# The code above are used inside Jupyter notebooks. If using other Python script, should write:
# from dotenv import load_dotenv
# load_dotenv()

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [17]:
import os
from glob import glob

# Write your code below.

PRICE_DATA = os.getenv("PRICE_DATA")
# or, PRICE_DATA="../../05_src/data/prices/"
# Read parquet files in the directory PRICE_DATA
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files) #?
dd_px.head()

,Date,Ticker,Close,High,Low,Open,Volume,Year
0,2000-01-03,AAPL,0.843076,0.847313,0.765877,0.789884,535796800.0,2000
64,2000-01-04,AAPL,0.771997,0.833191,0.762111,0.815304,512377600.0,2000
128,2000-01-05,AAPL,0.783293,0.832720,0.775762,0.781410,778321600.0,2000
192,2000-01-06,AAPL,0.715508,0.805889,0.715508,0.799299,767972800.0,2000
256,2000-01-07,AAPL,0.749401,0.760699,0.719275,0.726806,460734400.0,2000


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [19]:
# Write your code below.

import dask.dataframe as dd
dd_px1= dd.read_parquet(parquet_files).set_index("Ticker")
dd_px1.head()

,Date,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,
AAPL,2000-01-03,0.843076,0.847313,0.765877,0.789884,535796800.0,2000
AAPL,2018-05-11,44.718662,45.067231,44.448344,44.932073,104848800.0,2018
AAPL,2018-05-10,44.889389,44.967339,44.324847,44.346108,111957200.0,2018
AAPL,2018-05-09,44.256351,44.265798,43.750862,44.065022,92844800.0,2018
AAPL,2018-07-03,43.611305,44.566903,43.521198,44.528963,55819200.0,2018


In [ ]:
dd_shift = dd_px1.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)
# Given the parquet files only have variable Close, no Adj_Close, the code above add lags for Close only. Assuming both variables, the code would be:
# dd_shift = dd_px1.groupby('Ticker', group_keys=False).apply(
#    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1)),
#    lambda x: x.assign(Adj_Close_lag_1 = x['Adj_Close'].shift(1))
#)

In [ ]:
dd_feat = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    hi_lo_range = lambda x: x['High'] - x['Low']
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [33]:
# Write your code below.

# Convert the Dask data frame to a pandas data frame. 
pd_feat= dd_feat.compute()

# Calculate the moving average of `returns` using a window of 10 days. 
pd_feat["returns_ma_10"] = pd_feat["returns"].rolling(10).mean()

In [41]:
pd_feat.head

<bound method NDFrame.head of              Date       Close        High         Low        Open  \
Ticker                                                              
AAPL   2000-01-03    0.843076    0.847313    0.765877    0.789884   
AAPL   2000-01-04    0.771997    0.833191    0.762111    0.815304   
AAPL   2000-01-05    0.783293    0.832720    0.775762    0.781410   
AAPL   2000-01-06    0.715508    0.805889    0.715508    0.799299   
AAPL   2000-01-07    0.749401    0.760699    0.719275    0.726806   
...           ...         ...         ...         ...         ...   
ZBRA   2025-01-17  405.709991  407.290009  402.290009  406.040009   
ZBRA   2025-01-21  418.070007  419.850006  407.619995  407.619995   
ZBRA   2025-01-22  420.570007  427.760010  419.589996  425.239990   
ZBRA   2025-01-23  421.109985  422.290009  414.450012  417.619995   
ZBRA   2025-01-24  414.609985  420.079987  413.739990  419.059998   

             Volume  Year  Close_lag_1   Returns  hi_lo_range  returns_m

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

My answer: 

1. Was it necessary to convert to Pandas to calculate the moving average return?
No, it was not strictly necessary to convert to Pandas before computing the moving average. Dask supports rolling computations directly using .rolling().mean(), just like Pandas. You could have applied the rolling mean while still in Dask before calling .compute(), which would allow for better memory management when working with large datasets.

2. Would it have been better to do it in Dask? Why?
Yes, in most cases, it would be better to compute the moving average in Dask, especially for large datasets. Dask can handles large data efficiently hence get beeter performance.


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.